#**1. Crawling**

**Library**

In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
from google.colab import drive

In [24]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Base url**

In [47]:
Base_Url = "https://pta.trunojoyo.ac.id/c_search/byprod"

**Function**

In [56]:
def get_max_page(prodi_id):
    url = f"{Base_Url}/{prodi_id}"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, "html.parser")

    # cari tombol >> (last page)
    last_page = soup.select_one('ol.pagination a:contains("»")')
    if last_page and "href" in last_page.attrs:
        href = last_page["href"]
        # pecah URL -> ambil angka terakhir
        max_page = int(href.split("/")[-1])
        return max_page

    # fallback kalau pagination tidak ada
    return 1

In [57]:
def print_progress(prodi_id, prodi, current_page, total_pages):
    percent = (current_page / total_pages) * 100
    bar_length = 20
    filled_length = int(bar_length * current_page // total_pages)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    sys.stdout.write(f'\r[{prodi_id}] {prodi} - Page {current_page}/{total_pages} [{bar}] {percent:.2f}%')
    sys.stdout.flush()
    if current_page == total_pages:
        sys.stdout.write('\n')

**Function All Data**

In [61]:
def pta_manajemen():
    start_time = time.time()
    prodi_id = 7  # ID Prodi Manajemen
    data = {"judul": [], "abstrak_id": [], "prodi": []}

    max_page = get_max_page(prodi_id)
    print(f"🔍 Total halaman Prodi Manajemen: {max_page}")

    for j in range(1, max_page + 1):
        url = f"{Base_Url}/{prodi_id}/{j}"
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")
        jurnals = soup.select('li[data-cat="#luxury"]')

        isii = soup.select_one('div#begin')
        if not isii:
            continue
        prodi_full = isii.select_one('h2').text.strip()
        prodi = prodi_full.replace("Journal Jurusan ", "")

        for jurnal in jurnals:
            link_keluar = jurnal.select_one('a.gray.button')['href']
            response = requests.get(link_keluar)
            soup1 = BeautifulSoup(response.content, "html.parser")
            isi = soup1.select_one('div#content_journal')

            judul = isi.select_one('a.title').text.strip()
            paragraf = isi.select('p[align="justify"]')
            abstrak_id = paragraf[0].get_text(strip=True) if len(paragraf) > 0 else "N/A"

            data["judul"].append(judul)
            data["abstrak_id"].append(abstrak_id)
            data["prodi"].append(prodi)

        # ✅ Perbaikan: panggil dengan 4 argumen
        print_progress(prodi_id, prodi, j, max_page)

    # Simpan ke CSV
    df = pd.DataFrame(data)
    output_path = "/content/drive/MyDrive/PPW/output/pta_manajemen.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    # Hitung durasi
    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    print("\n✅ Seluruh data Manajemen berhasil dikumpulkan!")
    print(f"📈 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")

    return df

In [62]:
# --- Eksekusi ---
if __name__ == "__main__":
    prodi_id = 7
    pages = get_max_page(prodi_id)
    print(f"🔎 Cek Prodi Manajemen (ID {prodi_id}) punya total {pages} halaman.")
    confirm = input("Mau lanjut scraping semua data Manajemen? (y/n): ")
    if confirm.lower() == "y":
        df = pta_manajemen()
    else:
        print("❌ Scraping dibatalkan.")

🔎 Cek Prodi Manajemen (ID 7) punya total 207 halaman.
Mau lanjut scraping semua data Manajemen? (y/n): y
🔍 Total halaman Prodi Manajemen: 207
[7] Manajemen - Page 207/207 [████████████████████] 100.00%

✅ Seluruh data Manajemen berhasil dikumpulkan!
📈 Total entri: 1031
⏱️ Waktu eksekusi: 0 jam 31 menit 19 detik


In [65]:
manajemen_path = "/content/drive/MyDrive/PPW/output/pta_manajemen.csv"
manajemen = pd.read_csv(manajemen_path)

manajemen


,judul,abstrak_id,prodi
0,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",Manajemen
1,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,Manajemen
2,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,NaN,Manajemen
3,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,Manajemen
4,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak\r\nPenelitian ini menggunakan metode k...,Manajemen
...,...,...,...
1026,Analisis Cost Volume Profit Untuk Menentukan T...,ABSTRAK\nPenelitian ini bertujuan untuk menget...,Manajemen
1027,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...",Manajemen
1028,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,ABSTRAK\nTujuan dari penelitian ini adalah unt...,Manajemen
1029,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,Manajemen


#**2. Preprocessing**

In [67]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 12.7 MB/s eta 0:00:00


In [68]:
import pandas as pd
import re
from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

##**Sample Preprocessing**

In [70]:
sample_text = manajemen['abstrak_id'].dropna().iloc[0]
print("🔹 TEKS ASLI:\n", sample_text, "\n")

🔹 TEKS ASLI:
 ABSTRAK
Satiyah, Pengaruh Faktor-faktor Pelatihan dan Pengembangan Terhadap Produktivitas Kerja Dinas Kelautan dan Perikanan Bangkalan. Dibawah bimbingan Dra.Hj.S.Anugrahini Irawati,MM dan  Helmi Buyung Aulia,S,ST.SE,M.MT
Dalam upaya meningkatkan produktivitas kerja tidaklah mudah, oleh karena itu salah satu usaha agar produktivitas meningkat adalah menerapkan program pelatihan dan pengembangan sumber daya manusia (SDM) perlu dilaksanakan dalam instansi agar produktivitas yang tinggi dapat tercapai dengan meningkat kemampuan pegawai agar dapat bekerja secara efektif dan efisien. Dengan adanya pelatihan dan pengembnagan diharapkan pegawai juga mampu menyesuaikan diri dengan kebutuhan-kebutuhan baru atas sikap, tingkah laku keterampilan dan pengetahuan sesuai dengan tuntutan perubahan. Dengan adanya pelatihan dan pengembangan pegawai yang baik dapat mendukung terciptanya suasana kerja yang kondusif dalam instansi, sehingga dengan sendirinya produktivitasi kerja akan meningk

In [71]:
# --- 1. Case Folding ---
casefolded = sample_text.lower()
print("🔹 CASE FOLDING:\n", casefolded, "\n")

🔹 CASE FOLDING:
 abstrak
satiyah, pengaruh faktor-faktor pelatihan dan pengembangan terhadap produktivitas kerja dinas kelautan dan perikanan bangkalan. dibawah bimbingan dra.hj.s.anugrahini irawati,mm dan  helmi buyung aulia,s,st.se,m.mt
dalam upaya meningkatkan produktivitas kerja tidaklah mudah, oleh karena itu salah satu usaha agar produktivitas meningkat adalah menerapkan program pelatihan dan pengembangan sumber daya manusia (sdm) perlu dilaksanakan dalam instansi agar produktivitas yang tinggi dapat tercapai dengan meningkat kemampuan pegawai agar dapat bekerja secara efektif dan efisien. dengan adanya pelatihan dan pengembnagan diharapkan pegawai juga mampu menyesuaikan diri dengan kebutuhan-kebutuhan baru atas sikap, tingkah laku keterampilan dan pengetahuan sesuai dengan tuntutan perubahan. dengan adanya pelatihan dan pengembangan pegawai yang baik dapat mendukung terciptanya suasana kerja yang kondusif dalam instansi, sehingga dengan sendirinya produktivitasi kerja akan meni

In [72]:
# --- 2. Cleaning (hapus angka, tanda baca, karakter khusus) ---
cleaned = re.sub(r'[^a-zA-Z\s]', '', casefolded)
print("🔹 CLEANING:\n", cleaned, "\n")

🔹 CLEANING:
 abstrak
satiyah pengaruh faktorfaktor pelatihan dan pengembangan terhadap produktivitas kerja dinas kelautan dan perikanan bangkalan dibawah bimbingan drahjsanugrahini irawatimm dan  helmi buyung auliasstsemmt
dalam upaya meningkatkan produktivitas kerja tidaklah mudah oleh karena itu salah satu usaha agar produktivitas meningkat adalah menerapkan program pelatihan dan pengembangan sumber daya manusia sdm perlu dilaksanakan dalam instansi agar produktivitas yang tinggi dapat tercapai dengan meningkat kemampuan pegawai agar dapat bekerja secara efektif dan efisien dengan adanya pelatihan dan pengembnagan diharapkan pegawai juga mampu menyesuaikan diri dengan kebutuhankebutuhan baru atas sikap tingkah laku keterampilan dan pengetahuan sesuai dengan tuntutan perubahan dengan adanya pelatihan dan pengembangan pegawai yang baik dapat mendukung terciptanya suasana kerja yang kondusif dalam instansi sehingga dengan sendirinya produktivitasi kerja akan meningkat
tujuan penelitian 

In [73]:
# --- 3. Tokenisasi ---
tokens = cleaned.split()
print("🔹 TOKENISASI:\n", tokens, "\n")

🔹 TOKENISASI:
 ['abstrak', 'satiyah', 'pengaruh', 'faktorfaktor', 'pelatihan', 'dan', 'pengembangan', 'terhadap', 'produktivitas', 'kerja', 'dinas', 'kelautan', 'dan', 'perikanan', 'bangkalan', 'dibawah', 'bimbingan', 'drahjsanugrahini', 'irawatimm', 'dan', 'helmi', 'buyung', 'auliasstsemmt', 'dalam', 'upaya', 'meningkatkan', 'produktivitas', 'kerja', 'tidaklah', 'mudah', 'oleh', 'karena', 'itu', 'salah', 'satu', 'usaha', 'agar', 'produktivitas', 'meningkat', 'adalah', 'menerapkan', 'program', 'pelatihan', 'dan', 'pengembangan', 'sumber', 'daya', 'manusia', 'sdm', 'perlu', 'dilaksanakan', 'dalam', 'instansi', 'agar', 'produktivitas', 'yang', 'tinggi', 'dapat', 'tercapai', 'dengan', 'meningkat', 'kemampuan', 'pegawai', 'agar', 'dapat', 'bekerja', 'secara', 'efektif', 'dan', 'efisien', 'dengan', 'adanya', 'pelatihan', 'dan', 'pengembnagan', 'diharapkan', 'pegawai', 'juga', 'mampu', 'menyesuaikan', 'diri', 'dengan', 'kebutuhankebutuhan', 'baru', 'atas', 'sikap', 'tingkah', 'laku', 'ketera

In [74]:
# --- 4. Stopword Removal ---
stop_factory = StopWordRemoverFactory()
stopword = stop_factory.create_stop_word_remover()
no_stop = stopword.remove(" ".join(tokens))
print("🔹 STOPWORD REMOVAL:\n", no_stop.split(), "\n")

🔹 STOPWORD REMOVAL:
 ['abstrak', 'satiyah', 'pengaruh', 'faktorfaktor', 'pelatihan', 'pengembangan', 'produktivitas', 'kerja', 'dinas', 'kelautan', 'perikanan', 'bangkalan', 'dibawah', 'bimbingan', 'drahjsanugrahini', 'irawatimm', 'helmi', 'buyung', 'auliasstsemmt', 'upaya', 'meningkatkan', 'produktivitas', 'kerja', 'tidaklah', 'mudah', 'salah', 'satu', 'usaha', 'produktivitas', 'meningkat', 'menerapkan', 'program', 'pelatihan', 'pengembangan', 'sumber', 'daya', 'manusia', 'sdm', 'perlu', 'dilaksanakan', 'instansi', 'produktivitas', 'tinggi', 'tercapai', 'meningkat', 'kemampuan', 'pegawai', 'bekerja', 'efektif', 'efisien', 'adanya', 'pelatihan', 'pengembnagan', 'diharapkan', 'pegawai', 'mampu', 'menyesuaikan', 'diri', 'kebutuhankebutuhan', 'baru', 'atas', 'sikap', 'tingkah', 'laku', 'keterampilan', 'pengetahuan', 'sesuai', 'tuntutan', 'perubahan', 'adanya', 'pelatihan', 'pengembangan', 'pegawai', 'baik', 'dapat', 'mendukung', 'terciptanya', 'suasana', 'kerja', 'kondusif', 'instansi', '

In [75]:
# --- 5. Stemming ---
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()
stemmed = stemmer.stem(no_stop)
print("🔹 STEMMING:\n", stemmed.split(), "\n")

🔹 STEMMING:
 ['abstrak', 'satiyah', 'pengaruh', 'faktorfaktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'bawah', 'bimbing', 'drahjsanugrahini', 'irawatimm', 'helm', 'buyung', 'auliasstsemmt', 'upaya', 'tingkat', 'produktivitas', 'kerja', 'tidak', 'mudah', 'salah', 'satu', 'usaha', 'produktivitas', 'tingkat', 'terap', 'program', 'latih', 'kembang', 'sumber', 'daya', 'manusia', 'sdm', 'perlu', 'laksana', 'instansi', 'produktivitas', 'tinggi', 'capai', 'tingkat', 'mampu', 'pegawai', 'kerja', 'efektif', 'efisien', 'ada', 'latih', 'pengembnagan', 'harap', 'pegawai', 'mampu', 'sesuai', 'diri', 'kebutuhankebutuhan', 'baru', 'atas', 'sikap', 'tingkah', 'laku', 'terampil', 'tahu', 'sesuai', 'tuntut', 'ubah', 'ada', 'latih', 'kembang', 'pegawai', 'baik', 'dapat', 'dukung', 'cipta', 'suasana', 'kerja', 'kondusif', 'instansi', 'sendiri', 'produktivitas', 'kerja', 'tingkat', 'tuju', 'teliti', 'adalah', 'tahu', 'berapa', 'besar', 'pengaruh', 'faktorfaktor', 

##**Preprocessing keseluruhan**

In [82]:
# === Ambil kolom abstrak ===
abstrak_list = manajemen["abstrak_id"].dropna().astype(str).tolist()

###Inisialisasi stemmer & stopword remover

In [83]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

processed_words = []
sample_output = []  # untuk menampung contoh hasil

for idx, text in enumerate(abstrak_list[:5]):  # ambil 5 abstrak pertama dulu biar tidak panjang
    original = text
    text = text.lower()  # case folding
    text = re.sub(r'[^a-z\s]', ' ', text)  # hapus non-huruf
    tokens = text.split()  # tokenisasi
    tokens = [t for t in tokens if t not in stopwords]  # hapus stopword
    stemmed = [stemmer.stem(t) for t in tokens]  # stemming
    processed_words.extend(stemmed)

    sample_output.append({
        "abstrak_asli": original[:150] + "...",  # potong biar ringkas
        "tokens": tokens,
        "stemming": stemmed
    })

# tampilkan hasil sample preprocessing
import pandas as pd
pd.DataFrame(sample_output)

,abstrak_asli,tokens,stemming
0,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...","[abstrak, satiyah, pengaruh, faktor, faktor, p...","[abstrak, satiyah, pengaruh, faktor, faktor, l..."
1,Tujuan penelitian ini adalah untuk mengetahui ...,"[tujuan, penelitian, mengetahui, persepsi, bra...","[tuju, teliti, tahu, persepsi, brand, associat..."
2,Aplikasi nyata pemanfaatan teknologi informasi...,"[aplikasi, nyata, pemanfaatan, teknologi, info...","[aplikasi, nyata, manfaat, teknologi, informas..."
3,Abstrak\r\nPenelitian ini menggunakan metode k...,"[abstrak, penelitian, menggunakan, metode, kua...","[abstrak, teliti, guna, metode, kuantitatif, t..."
4,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...","[abstrak, aththaariq, pengaruh, kompetensi, do...","[abstrak, aththaariq, pengaruh, kompetensi, do..."


In [84]:
# === 4. Hitung frekuensi kata ===
counter = Counter(processed_words)

# === 5. Urutkan descending ===
sorted_freq = counter.most_common()

# === 6. Simpan ke CSV ===
freq_df = pd.DataFrame(sorted_freq, columns=["kata", "frekuensi"])
output_path = "/content/drive/MyDrive/PPW/output/frekuensi_kata_manajemen.csv"
freq_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ Frekuensi kata berhasil disimpan ke:", output_path)
print(freq_df.head(20))  # tampilkan 20 kata teratas

✅ Frekuensi kata berhasil disimpan ke: /content/drive/MyDrive/PPW/output/frekuensi_kata_manajemen.csv
             kata  frekuensi
0          teliti         24
1               x         23
2           kerja         19
3        variabel         18
4        pengaruh         15
5            guna         15
6      kompetensi         14
7   produktivitas         10
8         pegawai         10
9          faktor          9
10          besar          9
11         metode          9
12          nilai          9
13           data          9
14          latih          8
15          dinas          8
16       analisis          8
17        langgan          8
18        dimensi          8
19       akademik          8
